# Task 7: Machine Translation - Finetuned Model (Practical Pipeline)

## Objective
Finetune a pretrained model for production-quality EN↔NL round-translation.

## Why This Approach?
- The from-scratch model (main notebook) demonstrates understanding.
- This finetuned model provides practical translation quality for the pipeline.
- Combines educational learning with real-world application.

---

In [16]:
import numpy as np
from transformers import (
    MarianMTModel, 
    MarianTokenizer,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
    DataCollatorForSeq2Seq
)
from datasets import Dataset
import torch
from nltk.translate.bleu_score import corpus_bleu
import random

In [17]:
np.random.seed(42)
random.seed(42)
torch.manual_seed(42)

## Step 1: Load pretrained models

In [18]:
# Load pretrained models
print('Loading pretrained models...')

# EN -> NL model
model_en_nl_name = "Helsinki-NLP/opus-mt-en-nl"
tokenizer_en_nl = MarianTokenizer.from_pretrained(model_en_nl_name)
model_en_nl = MarianMTModel.from_pretrained(model_en_nl_name)

# NL -> EN model
model_nl_en_name = "Helsinki-NLP/opus-mt-nl-en"
tokenizer_nl_en = MarianTokenizer.from_pretrained(model_nl_en_name)
model_nl_en = MarianMTModel.from_pretrained(model_nl_en_name)

print('Models loaded')

Loading pretrained models...
Models loaded


## Step 2: Zero-shot test

In [19]:
def translate_marian(text, model, tokenizer, max_length=128):
    """ Translate using MarianMT model. """
    inputs = tokenizer(text, return_tensors='pt', padding=True, truncation=True, max_length=max_length)
    translated = model.generate(**inputs, max_length=max_length, num_beams=5)
    result = tokenizer.decode(translated[0], skip_special_tokens=True)
    return result

In [20]:
test_sentences = [
    'Hello, how are you?',
    'I love cooking.',
    'The weather is nice today.'
]

for sent in test_sentences:
    nl_translation = translate_marian(sent, model_en_nl, tokenizer_en_nl)
    en_translation = translate_marian(nl_translation, model_nl_en, tokenizer_nl_en)

    print(f'Original: {sent}')
    print(f'NL: {nl_translation}')
    print(f'Back EN: {en_translation}')
    print()

Original: Hello, how are you?
NL: Hallo, hoe gaat het?
Back EN: Hello, how are you?

Original: I love cooking.
NL: Ik hou van koken.
Back EN: I like to cook.

Original: The weather is nice today.
NL: Het weer is mooi vandaag.
Back EN: The weather's nice today.



## Step 3: Load Training Data


In [21]:
# Load the OpenSubtitles data
with open('OpenSubtitles.en-nl.en', 'r', encoding='utf-8') as f:
    en_sentences = [line.strip() for line in f.readlines()]

with open('OpenSubtitles.en-nl.nl', 'r', encoding='utf-8') as f:
    nl_sentences = [line.strip() for line in f.readlines()]

In [22]:
# Sample to manageable size
SAMPLE_SIZE = 50000 

# Sample randomly to get diverse data
indices = random.sample(range(len(en_sentences)), SAMPLE_SIZE)
en_sentences = [en_sentences[i] for i in sorted(indices)]
nl_sentences = [nl_sentences[i] for i in sorted(indices)]

print(f'Sampled {len(en_sentences)} sentence pairs')

Sampled 50000 sentence pairs


## Step 4: Prepare Dataset for Fine-Tuning

In [23]:
from sklearn.model_selection import train_test_split

# Split data
train_en, val_en, train_nl, val_nl = train_test_split(
    en_sentences, nl_sentences, test_size=0.1, random_state=42
)

# Create datasets for EN→NL
train_data_en_nl = Dataset.from_dict({
    'en': train_en,
    'nl': train_nl
})

val_data_en_nl = Dataset.from_dict({
    'en': val_en,
    'nl': val_nl
})

# Create datasets for NL→EN
train_data_nl_en = Dataset.from_dict({
    'nl': train_nl,
    'en': train_en
})

val_data_nl_en = Dataset.from_dict({
    'nl': val_nl,
    'en': val_en
})

print(f'Training samples: {len(train_en)}')
print(f'Validation samples: {len(val_en)}')

Training samples: 45000
Validation samples: 5000


## Step 5: Tokenization

In [24]:
def preprocess_function_en_nl(sentences):
    """ Tokenize for EN->NL model """
    inputs = tokenizer_en_nl(sentences['en'], truncation=True, max_length=128)
    targets = tokenizer_en_nl(sentences['nl'], truncation=True, max_length=128)
    inputs['labels'] = targets['input_ids']
    return inputs

def preprocess_function_nl_en(examples):
    """ Tokenize for NL->EN model """
    inputs = tokenizer_nl_en(examples['nl'], truncation=True, max_length=128)
    targets = tokenizer_nl_en(examples['en'], truncation=True, max_length=128)
    inputs['labels'] = targets['input_ids']
    return inputs

In [25]:
# Tokenize datasets
train_dataset_en_nl = train_data_en_nl.map(preprocess_function_en_nl, batched=True)
val_dataset_en_nl = val_data_en_nl.map(preprocess_function_en_nl, batched=True)

train_dataset_nl_en = train_data_nl_en.map(preprocess_function_nl_en, batched=True)
val_dataset_nl_en = val_data_nl_en.map(preprocess_function_nl_en, batched=True)

print('Datasets tokenized')

Map:   0%|          | 0/45000 [00:00<?, ? examples/s]

Map:   0%|          | 0/5000 [00:00<?, ? examples/s]

Map:   0%|          | 0/45000 [00:00<?, ? examples/s]

Map:   0%|          | 0/5000 [00:00<?, ? examples/s]

Datasets tokenized


## Step 6: Fine-Tuning EN->NL model

In [26]:
# Training arguments
training_args_en_nl = Seq2SeqTrainingArguments(
    output_dir='./machine_translation',
    eval_strategy='epoch',
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    save_total_limit=2,
    predict_with_generate=True,
    fp16=True,
    logging_dir='./machine_translation',
    logging_steps=100,
    save_strategy='epoch',
    load_best_model_at_end=True,
)

# Data collator
data_collator_en_nl = DataCollatorForSeq2Seq(tokenizer_en_nl, model=model_en_nl)

# Trainer
trainer_en_nl = Seq2SeqTrainer(
    model=model_en_nl,
    args=training_args_en_nl,
    train_dataset=train_dataset_en_nl,
    eval_dataset=val_dataset_en_nl,
    data_collator=data_collator_en_nl,
    tokenizer=tokenizer_en_nl,
)

/tmp/ipykernel_136159/3324928607.py:23: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Seq2SeqTrainer.__init__`. Use `processing_class` instead.
  trainer_en_nl = Seq2SeqTrainer(


In [27]:
print('Starting training...')
trainer_en_nl.train()

# Save finetuned model
model_en_nl.save_pretrained('./machine_translation/finetuned_en_nl')
tokenizer_en_nl.save_pretrained('./machine_translation/finetuned_en_nl')
print('EN->NL model finetuned and saved')

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.


Starting training...


Epoch,Training Loss,Validation Loss
1,1.575000,1.440435
2,1.331000,1.321118
3,1.228200,1.297882


/usr/local/lib/python3.11/dist-packages/transformers/modeling_utils.py:3922: UserWarning: Moving the following attributes in the config to the generation config: {'max_length': 512, 'num_beams': 4, 'bad_words_ids': [[67027]]}. You are seeing this warning because you've set generation parameters in the model config, as opposed to in the generation config.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/torch/utils/data/dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
/usr/local/lib/python3.11/dist-packages/torch/utils/data/dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
There were missing keys in the checkpoint model loaded: ['model.encoder.embed_tokens.weight', 'model.encoder.embed_positions.weight', 'model.decoder.embed_tokens.weight', 'model.deco

EN->NL model finetuned and saved


## Step 7: Fine-Tuning NL->EN Model

In [28]:
# Training arguments
training_args_nl_en = Seq2SeqTrainingArguments(
    output_dir='./machine_translation',
    eval_strategy='epoch',
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    save_total_limit=2,
    predict_with_generate=True,
    fp16=True,
    logging_dir='./machine_translation',
    logging_steps=100,
    save_strategy='epoch',
    load_best_model_at_end=True,
)

# Data collator
data_collator_nl_en = DataCollatorForSeq2Seq(tokenizer_nl_en, model=model_nl_en)

# Trainer
trainer_nl_en = Seq2SeqTrainer(
    model=model_nl_en,
    args=training_args_nl_en,
    train_dataset=train_dataset_nl_en,
    eval_dataset=val_dataset_nl_en,
    data_collator=data_collator_nl_en,
    tokenizer=tokenizer_nl_en,
)

/tmp/ipykernel_136159/3447355004.py:23: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Seq2SeqTrainer.__init__`. Use `processing_class` instead.
  trainer_nl_en = Seq2SeqTrainer(


In [29]:
print('Starting training...')
trainer_nl_en.train()

# Save finetuned model
model_nl_en.save_pretrained('./machine_translation/finetuned_nl_en')
tokenizer_nl_en.save_pretrained('./machine_translation/finetuned_nl_en')
print('NL->EN model finetuned and saved')

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.


Starting training...


/usr/local/lib/python3.11/dist-packages/torch/utils/data/dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Epoch,Training Loss,Validation Loss
1,1.670700,1.511279
2,1.492600,1.426082
3,1.332000,1.407412


/usr/local/lib/python3.11/dist-packages/transformers/modeling_utils.py:3922: UserWarning: Moving the following attributes in the config to the generation config: {'max_length': 512, 'num_beams': 6, 'bad_words_ids': [[67027]]}. You are seeing this warning because you've set generation parameters in the model config, as opposed to in the generation config.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/torch/utils/data/dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
/usr/local/lib/python3.11/dist-packages/torch/utils/data/dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
There were missing keys in the checkpoint model loaded: ['model.encoder.embed_tokens.weight', 'model.encoder.embed_positions.weight', 'model.decoder.embed_tokens.weight', 'model.deco

NL->EN model finetuned and saved


## Step 8: Evaluation

In [30]:
def evaluate_marian_bleu(en_texts, nl_texts, direction='en_to_nl', n_samples=500):
    """ Evaluate BLEU for MarianMT models """
    references = []
    hypotheses = []
    
    samples = min(n_samples, len(en_texts))
    
    if direction == 'en_to_nl':
        model, tokenizer = model_en_nl, tokenizer_en_nl
        sources = en_texts[:samples]
        targets = nl_texts[:samples]
    else:
        model, tokenizer = model_nl_en, tokenizer_nl_en
        sources = nl_texts[:samples]
        targets = en_texts[:samples]
    
    print(f'Evaluating {samples} samples for {direction}...')
    
    for i in range(samples):
        ref_tokens = targets[i].strip().split()
        translation = translate_marian(sources[i], model, tokenizer)
        hyp_tokens = translation.strip().split()
        
        references.append([ref_tokens])
        hypotheses.append(hyp_tokens)
        
        if (i + 1) % 100 == 0:
            print(f'  Progress: {i+1}/{samples}')
    
    bleu = corpus_bleu(references, hypotheses)
    return bleu * 100

In [31]:
# Evaluate finetuned models
bleu_en_nl = evaluate_marian_bleu(val_en, val_nl, direction='en_to_nl')
bleu_nl_en = evaluate_marian_bleu(val_en, val_nl, direction='nl_to_en')

print(f'BLEU EN->NL: {bleu_en_nl:.2f}')
print(f'BLEU NL->EN: {bleu_nl_en:.2f}')
print(f'Average: {(bleu_en_nl + bleu_nl_en) / 2:.2f}')

Evaluating 500 samples for en_to_nl...
  Progress: 100/500
  Progress: 200/500
  Progress: 300/500
  Progress: 400/500
  Progress: 500/500
Evaluating 500 samples for nl_to_en...
  Progress: 100/500
  Progress: 200/500
  Progress: 300/500
  Progress: 400/500
  Progress: 500/500
BLEU EN->NL: 21.17
BLEU NL->EN: 23.25
Average: 22.21
